# First Machine Learning Classifier – Decision Tree

**Objective:** Build and evaluate a Decision Tree classification model using Scikit-Learn to predict whether a Titanic passenger survived.

**Dataset:** Titanic dataset from Seaborn.

**What you will learn:**
- Load and explore a real dataset
- Prepare features and target
- Split data into training and testing sets
- Build a preprocessing + Decision Tree pipeline
- Train and make predictions
- Evaluate using accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC
- Visualize the decision tree and feature importance

Run all cells from top to bottom.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
)

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## 2. Load the Titanic dataset

In [ ]:
# Seaborn provides the classic Titanic dataset
df = sns.load_dataset("titanic")

print("Dataset shape:", df.shape)
display(df.head())

## 3. Explore the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))

print("\nTarget distribution (survived):")
display(df["survived"].value_counts().rename(index={0: "Did not survive", 1: "Survived"}).to_frame("Passenger count"))

## 4. Select features and target

In [ ]:
# Predict whether a passenger survived (1) or did not survive (0).
# Exclude identifiers, names, and columns that would not be used for this beginner model.
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

X = df[features].copy()
y = df[target].copy()

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

print("Features:", features)
print("Target:", target)
print("X shape:", X.shape)
print("y shape:", y.shape)

## 5. Split into training and testing data

In [ ]:
# Stratify preserves the survival-class proportion in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTesting target proportions:")
print(y_test.value_counts(normalize=True).round(3))

## 6. Build preprocessing and Decision Tree pipeline

In [ ]:
# Numeric columns: fill missing values with the median.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical columns: fill missing values with the most frequent value,
# then convert categories into numeric indicator columns.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Limit tree depth to help control overfitting in this first model.
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ))
])

print("Preprocessing and Decision Tree pipeline created!")

## 7. Train the model

In [ ]:
model.fit(X_train, y_train)
print("Model training completed!")

## 8. Make predictions

In [ ]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred
})
display(results.head(15))

## 9. Evaluate model performance

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics.style.format({"Score": "{:.3f}"}))
print("\nDetailed classification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Did not survive", "Survived"],
    zero_division=0
))

## 10. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Decision Tree – Confusion Matrix")
plt.tight_layout()
plt.show()

## 11. ROC curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title("Decision Tree – ROC Curve")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Feature importance

In [ ]:
# Retrieve transformed feature names after one-hot encoding
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
      .sort_values("Importance", ascending=False)
)

display(importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Decision Tree Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 13. Visualize the Decision Tree

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    model.named_steps["classifier"],
    feature_names=feature_names,
    class_names=["Did not survive", "Survived"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Titanic Decision Tree Classifier")
plt.show()

## 14. Predict for a new passenger

In [ ]:
# Example passenger details. Change these values to test other cases.
new_passenger = pd.DataFrame([{
    "pclass": 3,
    "sex": "male",
    "age": 25,
    "sibsp": 0,
    "parch": 0,
    "fare": 7.25,
    "embarked": "S"
}])

prediction = model.predict(new_passenger)[0]
probability = model.predict_proba(new_passenger)[0, 1]

print("Predicted class:", "Survived" if prediction == 1 else "Did not survive")
print(f"Model estimated survival probability: {probability:.2%}")

## 15. Final model summary

In [ ]:
print("========== DECISION TREE MODEL REPORT ==========")
print("Algorithm: Decision Tree Classifier")
print("Dataset: Titanic")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")
print("================================================")

## Conclusion

In this project, a Decision Tree Classifier was trained using Scikit-Learn to predict Titanic passenger survival. Missing values were handled inside a preprocessing pipeline, categorical variables were one-hot encoded, and the data was divided into training and testing sets.

The model was evaluated using accuracy, precision, recall, F1-score, ROC-AUC, and a confusion matrix. Feature importance and a tree plot were used to interpret the model.

**Important:** These metrics measure performance on the held-out test set, not guaranteed real-world performance. The model demonstrates a beginner classification workflow; it does not establish causal reasons for survival.